In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), # 좌우 반전
    transforms.RandomCrop(32, padding=4), # 살짝 이동 + 확대 효과
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(
    root = "./data",
    train = True,
    download= True,
    transform = train_transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform = test_transform
)

train_loader = DataLoader(train_dataset, batch_size= 128, shuffle = True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle= False, num_workers=0)

class CIFAR_CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32,32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2), # 32 -> 16

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2), # 16 -> 8

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2), # 8 -> 4
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    
# 모델, 손실함수, 옵티마이저 준비
model = CIFAR_CNN()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 학습 코드
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # GPU or CPU Mac은 NVIDIA CUDA를 지원안해서 CPU
model = model.to(device) # 모델을 CPU에 적용시키는 작업 / 데이터와 모델은 같은 장치에 있어야함
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5) # 초반 빠르게 학습 -> 후반 안정적으로 미세조정 / epoch 끝날때 scheduler.step()
                                                                               # optimizer: 어떤 옵티마이저의 Lr을 조정할지 , step_size: lr 몇번마다 조정해줄건지, gamma: lr을 얼마만큼 줄일건지
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau( 
#     optimizer,
#     mode='max',
#     factor= 0.5,
#     patience=4,
#     threshold=0.001) 
# 성능 멈추면 자동으로 lr을 줄여준다. StepLR보다 똑똑한편
# mode= 'max': metrics값을 accuracy로 주는데 이건 클수록 좋으니까 Max값을 보는 모드로
# factor: 얼마나 줄일거냐, lr = lr x factor
# patience: 성능이 안좋아진 상태를 얼마나 볼 것 인지, 최고점에서 n번떨어져도 참아라.
# threshold: 0.001 이하 변화는 변화 없는것으로 판단하고 lr을 감소시키지 말아라.

epochs = 3 # 반복 횟수 / 훈련 데이터 전체를 한바퀴 다보는게 1 epoch
best_acc = 0 # 최고 좋은 epoch를 찾는 변수
patience = 1
counter = 0

for epoch in range(epochs): # epoch 횟수만큼 학습 반복
    model.train() # 모델을 학습모드로 전환 training
    train_loss = 0 # epoch동안 손실을 누적해서 기록하기 위한 변수

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch+1}, Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}")

    # for images, labels in train_loader: # train 데이터를 batch단위로 하나씩 꺼내오기
    #     images, labels = images.to(device), labels.to(device) # 데이터들도 model과 같은 디바이스로 옮겨주기

    #     optimizer.zero_grad() # 이전 batch에서 계산된 기울기를 초기화 / Pytorch는 기울기를 계속 누적하는데 초기화를 안하면 이전것과 이번것이 섞여버림 / 이번 batch 기준으로 업데이트
    #     outputs = model(images) # 모델이 이미지를 보고 예측값 생성 / ex) (64,10).. 64장의 이미지 각 이미지마다 10개의 숫자에 대한 점수가 10개
    #     loss = criterion(outputs, labels) # 모델 예측이 정답과 얼마나 다른지 계산 / 예를들어 정답이 2인데 모델이 2에 높은점수를 주면 loss가 낮고, 다른수에 높은점수를 주면 loss가 높다.
    #     loss.backward() # 위에서 계산한 오차를 바탕으로 어떤 가중치를 얼마나 고쳐야 하는지 계산하는 작업 / 틀린 원인을 뒤로 거슬러 올라가며 각 가중치의 책임을 계산 (정답 비교-> 오차 계산-> 뒤로 전달)
    #     optimizer.step() # 방금 계산한 기울기(gradient)를 이용해 실제로 가중치를 수정 / loss.backward() = 오답 분석, optimizer.step() = 실제 수정

    #     train_loss += loss.item() # 현재 batch의 loss를 숫자로 꺼내서 변수에 누적 / loss.item()은 tensor형태(파이썬숫자)라서 출력용 숫자로 변형하기위해 .item()을 사용
    avg_train_loss = train_loss / len(train_loader)

    model.eval() # 모델을 평가모드로 전환
    correct = 0 # 검증 데이터에서 맞춘 개수 카운트하려는 변수
    total = 0 # 검증 데이터의 전체 개수 카운트하는 변수

    with torch.no_grad(): # 검증할 때에는 gradient 계산하지 말라는 명령어 / 검증할때에는 학습 안하고 평가만 하니 역전파용 계산이 무쓸모.
        for images, labels in test_loader: # 검증 데이터도 batch 단위로 꺼내오기 단, 이곳에선 train처럼 학습하지 않고 평가만 함.
            images, labels = images.to(device), labels.to(device)
            outputs = model(images) # 검증 데이터에 대해 모델의 예측
            _, predicted = torch.max(outputs, 1) # torch.max(outputs, 1)은 가장 큰값, 그 위치(index)를 반환하는데 우리는 index만 필요하니 앞의 값은 버려줌.
                                                 # 1의 의미는 Pytorch에서 0은 세로방향(행) 1이 가로방향(열)을 의미
                                                 # 코드 의미: 각 행(= 각 이미지)에 대해 가장 큰 값과 그 위치를 찾아라 / 각 행에서 가로방향으로 최대값 찾기 > 열방향으로 움직이며 계산
                                                 # dim = 그 축을 없애면서 연산, dim=1 > 열 방향으로 계산 > 행마다 결과
                                                 # dim=0 > 행 방향으로 계산 > 열마다 결과 / dim은 어느방향으로 훑을지!

            total += labels.size(0) # 이번 batch의 데이터 개수를 total에 더해주기. batch size가 64면 한번에 64추가
            correct += (predicted == labels).sum().item() # 이번 batch에서 맞춘 개수 세기 >> 예측과 정답을 비교해서 .sum()으로 True를 1처럼 세서 계산 후 .item()으로 숫자로 바꿔줌

        accuracy = correct / total # 전체 중에서 맞춘 비율

        if accuracy > best_acc: # best_acc변수에 계속 최고 accuracy를 넣어줌
            best_acc = accuracy 
            torch.save(model.state_dict(), "best_model.pth") # 현재 모델의 학습된 가중치를 파일로 저장, 모델에 학습된 모든 값을 파이썬 객체 파일로 저장하는 함수, .pth는 파일 확장자+저장할 파일이름
                                                             # state_dict는 모델이 학습한 모든 지식(뇌)라고 생각하면 됨. 
            counter = 0 # 카운터 초기화
        else:
            counter += 1 # 최고 accuracy보다 안좋은 accuracy가 나온 횟수를 카운트

        print(f"Epoch {epoch+1}/{epochs}, Train loss: {avg_train_loss:.4f}, Valid Accuracy: {accuracy:.4f}") # 현재 epoch의 학습상태 출력
        
        scheduler.step() # StepLR은 epoch기준으로 자동감소해서 metrics를 안줘도 되지만 ReduceLROnPlateau는 성능을 보고 lr을 줄이는 방식이라 metrics값 즉 accuracy or loss를 주어줘야한다
        
        if counter >= patience:
            print("Early Stopping!")
            break
        
        # 추후에 파일 불러와서 사용할 때
        # model = BetterCNN()  구조 먼저 만들고
        # model.load_state_dict(torch.load("best_model.pth")) 파일 불러오기
        # model.eval() 시험 모드 시작

100.0%
/Users/jin/Desktop/AI-STUDY/.venv/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Epoch 1, Batch 0/391, Loss: 2.3514
Epoch 1, Batch 100/391, Loss: 1.4709
Epoch 1, Batch 200/391, Loss: 1.2875
Epoch 1, Batch 300/391, Loss: 1.2347
Epoch 1/3, Train loss: 1.4879, Valid Accuracy: 0.5490
Epoch 2, Batch 0/391, Loss: 1.4163
Epoch 2, Batch 100/391, Loss: 1.1364
Epoch 2, Batch 200/391, Loss: 1.0387
Epoch 2, Batch 300/391, Loss: 0.8096
Epoch 2/3, Train loss: 1.0658, Valid Accuracy: 0.6164
Epoch 3, Batch 0/391, Loss: 0.8221
Epoch 3, Batch 100/391, Loss: 0.8750
Epoch 3, Batch 200/391, Loss: 0.9986
Epoch 3, Batch 300/391, Loss: 0.9905
Epoch 3/3, Train loss: 0.9135, Valid Accuracy: 0.7113
